In [ ]:
using DrWatson
@quickactivate "IonChannels"

using IonChannelTools

In [ ]:
#modified package functions to accomodate additional matrix parameter
function evolvedistNE(G,L,dt,mu0,epsilon)
    mu = Array{Float64}(undef,0,length(mu0))
    mu = [mu; mu0] #array of state dist. begins with initial mu0
    for i = 1:length(L)
        Ga = G(L[i],epsilon) #calculate transition rate matrix for given driving L
        newmu = transpose(mu[end,:]) * exp(Ga*dt) #new state distribution
        c = findall(x -> x < 0, newmu) #indices of all negative values in newmu
        newmu[c] .= 1e-20

        mu = [mu ; newmu/sum(newmu)] #normalizes to sum(prob)=1 so no lil artifacts hopefully
        end
    return mu[2:end,:] #returns all but the first line (so array size matches L)
    end

In [ ]:
include(srcdir("WLMSR-NE.jl"))

In [ ]:
eps = 1e-4
G1 = Gmatrix(40,eps) #

function Q_hk(G)
    Qhk = Array{Float64,2}(undef,0,2)
    ss = IonChannelTools.steadystate(G)
    for i in 1:size(G)[1]
        for j in i:size(G)[2] #no duplicates i-j j-i 
            if i!=j && !isnan(log((ss[i]*G[i,j])/(ss[j]*G[j,i]))) 
                #no self transitions and excludes any forbidden transitions log0/0=NaN
                Qhk = [Qhk; (i,j) log((ss[i]*G[i,j])/(ss[j]*G[j,i]))]
            end
        end
    end
    #replace!(Qhk,NaN=>0) #log 0/0 = 0 #not needed anymore
    Qhk[findall(x->abs(x)<1e-10,Qhk[:,2]),2] .= 0 #artifacts
    return Qhk
end

Q_hk(G1)

##### 

**Change in steady state occupancy**

In [ ]:
using LinearAlgebra
step = 100
eps = logrange(1e-10,1,step)
alphas = range(-40,100,step)
ss = zeros(step,step,5)
for i in 1:step, j in 1:step
    ss[i,j,:] = IonChannelTools.steadystate(Gmatrix(alphas[j],eps[i]))
end
ss

In [ ]:
dot = zeros(step,step)
ss0 = Array{Float64}(undef,step,5)
for j in 1:step
    ss0[j,:] = IonChannelTools.steadystate(Gmatrix(alphas[j],0))
end
for i in 1:step, j in 1:step
    dot[i,j] = 1-LinearAlgebra.dot(LinearAlgebra.normalize(ss[i,j,:]),LinearAlgebra.normalize(ss0[j,:]))
end
dot[findall(x->abs(x)<1e-10,dot)] .= 0 #artifacts
dot

In [ ]:
using Plots
heatmap(alphas,eps,dot,
    title="orthogonality of steady states",
    yscale=:log10,
    c=:dense)#,ratio=:equal)

In [ ]:
clim = 0.3
heatmap(alphas,eps,ss[:,:,4].-LinearAlgebra.transpose(ss0[:,4]),
    title="difference in ss O occupancy",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:balance,
    clims = (-clim,clim))


In [ ]:
heatmap(alphas,eps,ss[:,:,5].-LinearAlgebra.transpose(ss0[:,5]),
    title="difference in ss I occupancy",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:balance,
    clims = (-clim,clim))


In [ ]:
heatmap(alphas,eps,ss[:,:,3].-LinearAlgebra.transpose(ss0[:,3]),
    title="difference in ss C3 occupancy",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:balance,
    clims = (-clim,clim))


In [ ]:
heatmap(alphas,eps,ss[:,:,2].-LinearAlgebra.transpose(ss0[:,2]),
    title="difference in ss C2 occupancy",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:balance,
    clims = (-clim,clim))


In [ ]:
heatmap(alphas,eps,ss[:,:,1].-LinearAlgebra.transpose(ss0[:,1]),
    title="difference in ss C1 occupancy",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:balance,
    clims = (-clim,clim))


**Pairwise housekeeping heat**

In [ ]:
Qhk = zeros(step,step,5)
Qhk_index = Q_hk(Gmatrix(alphas[1],eps[1]))[:,1]
for i in 1:step, j in 1:step
    Qhk[i,j,:] = Q_hk(Gmatrix(alphas[j],eps[i]))[:,2]
end
Qhk_index

In [ ]:
heatmap(alphas,eps,Qhk[:,:,5],
    title="Qhk for O-->I transition",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:dense)

O-->I transition is -infty

In [ ]:
heatmap(alphas,eps,Qhk[:,:,3],
    title="Qhk for C3-->I transition",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:dense)

In [ ]:
heatmap(alphas,eps,Qhk[:,:,1],
    title="Qhk for C1-->C2 transition",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:dense)

In [ ]:
heatmap(alphas,eps,Qhk[:,:,2],
    title="Qhk for C2-->C3 transition",
    xlabel="mV",
    ylabel="epsilon",
    yscale=:log10,
    color=:dense)